In [1]:
import pandas as pd
# import pathlib as pl


import os


from __future__ import annotations

from typing import Dict, Any, Union
import math

from pathlib import Path




In [2]:
# Directories

RAW_DATA_DIR = Path("../landing_zone/")
READY_TO_UPLOAD_FILES_DIR = Path("../ready_to_upload/")

RAW_DATA_DIR = RAW_DATA_DIR / "2026" / "KIA"
SAMPLE_DATA_FILE_PATH = RAW_DATA_DIR / "Accessory Guide - February26.xlsx"
READY_TO_UPLOAD_DIR = READY_TO_UPLOAD_FILES_DIR / "2026" / "KIA"

In [3]:
# Sheet name keywords that indicate NON-DATA sheets
# Case-insensitive, substring match
EXCLUDED_SHEET_NAME_KEYWORDS = [
    "Luxwood",
]


In [4]:
def is_valid_data_sheet(sheet_name: str) -> bool:
    """
    Determine whether a sheet name represents an actual data sheet.

    Returns False if the sheet name contains any excluded keyword.
    """
    normalized = sheet_name.lower()

    return not any(
        keyword in normalized
        for keyword in EXCLUDED_SHEET_NAME_KEYWORDS
    )



def drop_fully_empty_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop columns that are entirely empty (all values NaN).

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame AFTER header promotion.

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame with fully empty columns removed.

    Raises
    ------
    ValueError
        If df is None or empty.
    """

    if df is None or df.empty:
        raise ValueError("Cannot clean DataFrame: DataFrame is empty or None")

    # Drop columns where ALL values are NaN
    cleaned_df = df.dropna(axis=1, how="all")

    return cleaned_df

In [5]:


def extract_sheet_meta_data_from_columns(
    df: pd.DataFrame,
    sheet_name: str,
) -> Dict[str, Any]:
    """
    Extract metadata from sheet columns.

    IMPORTANT:
    - This function REFUSES to run on ignored sheets.
    """

    # ---- hard guard: ignored sheets are forbidden
    if not is_valid_data_sheet(sheet_name):
        raise ValueError(
            f"Metadata extraction attempted on ignored sheet: '{sheet_name}'"
        )

    if df.empty:
        raise ValueError(f"Sheet '{sheet_name}' is empty")

    # -----------------------------
    # Filter usable columns
    # -----------------------------
    valid_columns = [
        col for col in df.columns
        if "unnamed" not in str(col).lower()
    ]

    if not valid_columns:
        raise ValueError(
            f"Sheet '{sheet_name}' has no usable columns after filtering 'Unnamed'"
        )

    # -----------------------------
    # 1) Model name from column 0
    # -----------------------------
    model_name = str(valid_columns[0]).strip()

  

    # -----------------------------
    # 2) Extract L_rate from column values
    # -----------------------------
    L_rate = "not found"

    for col in valid_columns:
        value = col

        if value is None or (isinstance(value, float) and math.isnan(value)):
            continue

        if isinstance(value, (int, float)):
            L_rate = value
            break

        if isinstance(value, str):
            try:
                L_rate = float(value.strip())
                break
            except ValueError:
                continue

    return {
        "model_name": model_name,
        "L_rate": L_rate,
    }


In [6]:



def check_l_rate_consistency(meta_data: Dict[str, Dict[str, Any]]) -> bool:
    """
    Check whether all L_rate values are the same across all models.

    Parameters
    ----------
    meta_data : dict
        Dictionary of per-sheet metadata in the form:
        {
            "SheetName": {
                "model_name": str,
                "L_rate": float | int | "not found"
            },
            ...
        }

    Returns
    -------
    bool
        True if all L_rate values are identical.

    Raises
    ------
    ValueError
        If meta_data is empty, malformed, contains missing L_rate values,
        or if L_rate values differ across models.
    """

    if not isinstance(meta_data, dict) or not meta_data:
        raise ValueError("meta_data must be a non-empty dictionary")

    l_rates = {}
    for sheet_name, data in meta_data.items():
        if not isinstance(data, dict):
            raise ValueError(f"Metadata for '{sheet_name}' is not a dictionary")

        if "L_rate" not in data:
            raise ValueError(f"Missing L_rate for sheet '{sheet_name}'")

        l_rate = data["L_rate"]

        if l_rate == "not found":
            raise ValueError(f"L_rate not found for sheet '{sheet_name}'")

        if not isinstance(l_rate, (int, float)):
            raise ValueError(
                f"Invalid L_rate type for sheet '{sheet_name}': {type(l_rate).__name__}"
            )

        l_rates[sheet_name] = l_rate

    unique_rates = set(l_rates.values())

    if len(unique_rates) > 1:
        raise ValueError(
            "L_rate values are not consistent across models: "
            f"{l_rates}"
        )

    return True

In [7]:
def promote_first_row_to_header(df: pd.DataFrame) -> pd.DataFrame:
    """
    Promote row 0 to become column headers and drop it from the data.

    Raises
    ------
    ValueError if the DataFrame has fewer than 2 rows
    """

    if df.shape[0] < 2:
        raise ValueError("Cannot promote first row to header: insufficient rows")

    new_header = df.iloc[0].astype(str).str.strip()
    df = df.iloc[1:].reset_index(drop=True)
    df.columns = new_header
    return df

   

In [8]:

# from typing import Dict, Union

# import pandas as pd


def load_excel_sheets_to_dict(
    excel_file_path: Union[str, Path]
) -> Dict[str, object]:
    """
    Load ONLY valid data sheets from an Excel file.
    Ignored sheets are excluded from BOTH data and metadata.
    Applies header promotion and column cleanup.
    """

    path = Path(excel_file_path).expanduser().resolve()

    if not path.exists():
        raise FileNotFoundError(f"Excel file not found: {path}")

    if not path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    valid_extensions = {".xls", ".xlsx", ".xlsm", ".xlsb", ".ods"}
    if path.suffix.lower() not in valid_extensions:
        raise ValueError(f"Unsupported Excel file type: {path.suffix}")

    try:
        excel = pd.ExcelFile(path)
    except Exception as exc:
        raise RuntimeError(f"Failed to open Excel file: {path}") from exc

    result: Dict[str, object] = {}
    # meta_data: Dict[str, Dict[str, object]] = {}

    for raw_sheet_name in excel.sheet_names:
        clean_sheet_name = raw_sheet_name.strip()

        # # ---- skip ignored sheets
        # if not is_valid_data_sheet(clean_sheet_name):
        #     continue

        if clean_sheet_name in result:
            raise ValueError(
                f"Duplicate sheet name after stripping: '{clean_sheet_name}'"
            )

        try:
            df_raw = excel.parse(sheet_name=raw_sheet_name)
        except Exception as exc:
            raise RuntimeError(
                f"Failed to parse sheet '{raw_sheet_name}' from {path}"
            ) from exc


        result[clean_sheet_name] = df_raw

    if not result:
        raise ValueError(
            f"No valid data sheets found in {path}. "
            f"Excluded keywords: {EXCLUDED_SHEET_NAME_KEYWORDS}"
        )

    return result

Data Exploratory Analysis

In [9]:
# Loaded data 
SAMPLE_DATA_FILE_PATH = RAW_DATA_DIR / "KIA 202604 Accessories Application Chart.xlsx"

data_dict = load_excel_sheets_to_dict(SAMPLE_DATA_FILE_PATH)


c:\Users\paxm\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Slicer List extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\paxm\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Slicer List extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\paxm\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Slicer List extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\paxm\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Slicer List extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\paxm\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Slicer List extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Us

## Results:

1. All essential data elements available in the file? -> Most of them are there, except we are missing remarks in french.
2.  